# 추가 백엔드-2 (PASS/FAIL 집계) - Jupyter Notebook

본 노트북은 첨부된 **「2026-01-29 추가 백엔드-2 설계」** 사양을 그대로 구현합니다.  
- DB에서 `prod_day` + `shift_type(day/night)` 기준으로 시간창을 만들고  
- `a1_fct_vision_testlog_txt_processing_history.fct_vision_testlog_txt_processing_history` 를 스캔하여  
- `station`별 PASS/Total/PASS%를 **FCT1~4, Vision1~2** 컬럼으로 출력합니다.  

또한 `barcode_information`의 **18번째 문자**를 `g_production_film.remark_info` 기준으로 **품번(pn) + remark(PD/Non-PD)** 로 분류하는 유틸도 포함합니다.


In [1]:
# =========================
# 0) 설치/환경 (필요 시)
# =========================
# !pip -q install sqlalchemy psycopg2-binary pandas python-dateutil

import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime, date, time, timedelta
from typing import Tuple, Optional, Dict, Any


In [2]:
# =========================
# 1) DB 설정 (사양 그대로)
# =========================
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "leejangwoo1!",
}

def get_engine():
    # pool은 노트북 특성상 과도하게 키우지 않음
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    return create_engine(url, pool_size=1, max_overflow=0, pool_pre_ping=True)

engine = get_engine()

# 연결 테스트
with engine.connect() as conn:
    conn.execute(text("SELECT 1")).scalar()
print("DB connect OK")


DB connect OK


In [3]:
# =========================
# 2) 주/야간 시간창 (KST 로컬 기준)
# =========================
# 사양:
# day   : [D-DAY] 08:30:00 ~ 20:29:59 (양끝 포함)
# night : [D-DAY] 20:30:00 ~ 23:59:59 + [D+1] 00:00:00 ~ 08:29:59 (양끝 포함)
#
# 구현:
# - SQL에서는 inclusive 처리하기 위해 [start, end] 를 [start, end+1sec) 로 변환해 사용
#   (즉, end 포함을 >= start AND < end_exclusive 로 안전하게 처리)

DAY_START   = time(8, 30, 0)
DAY_END     = time(20, 29, 59)
NIGHT_START = time(20, 30, 0)
NIGHT_END   = time(8, 29, 59)

def parse_prod_day(prod_day: str) -> date:
    # prod_day: 'YYYYMMDD'
    return datetime.strptime(prod_day, "%Y%m%d").date()

def shift_window_inclusive(prod_day: str, shift_type: str) -> Tuple[datetime, datetime]:
    d = parse_prod_day(prod_day)
    if shift_type.lower() == "day":
        start = datetime.combine(d, DAY_START)
        end   = datetime.combine(d, DAY_END)
    elif shift_type.lower() == "night":
        start = datetime.combine(d, NIGHT_START)
        end   = datetime.combine(d + timedelta(days=1), NIGHT_END)
    else:
        raise ValueError("shift_type must be 'day' or 'night'")
    return start, end

def shift_window_halfopen(prod_day: str, shift_type: str) -> Tuple[datetime, datetime]:
    start, end_incl = shift_window_inclusive(prod_day, shift_type)
    end_excl = end_incl + timedelta(seconds=1)
    return start, end_excl

# 예시
for st in ["day", "night"]:
    s, e = shift_window_halfopen("20260127", st)
    print(st, "=>", s, "~", e, "(half-open)")


day => 2026-01-27 08:30:00 ~ 2026-01-27 20:30:00 (half-open)
night => 2026-01-27 20:30:00 ~ 2026-01-28 08:30:00 (half-open)


In [4]:
# =========================
# 3) barcode_information 18번째 문자 → pn/remark 분류
# =========================
# 사양:
# - 스키마: g_production_film, 테이블: remark_info
# - barcode_information의 18번째 문자(key)로 매핑
# - 예: '...J...' => key='J' => pn=35930927, remark=PD

REMARK_SCHEMA = "g_production_film"
REMARK_TABLE  = "remark_info"

def load_remark_map(engine) -> Dict[str, Dict[str, str]]:
    sql = text(f'''
        SELECT barcode_information::text AS key, pn::text AS pn, remark::text AS remark
        FROM {REMARK_SCHEMA}.{REMARK_TABLE}
    ''')
    with engine.connect() as conn:
        df = pd.read_sql(sql, conn)
    # key의 공백/대소문자 안정화
    df["key"] = df["key"].str.strip()
    mp = {r["key"]: {"pn": r["pn"], "remark": r["remark"]} for _, r in df.iterrows()}
    return mp

remark_map = load_remark_map(engine)
remark_map


{'C': {'pn': '35643009', 'remark': 'Non-PD'},
 'P': {'pn': '35643010', 'remark': 'Non-PD'},
 '1': {'pn': '35654264', 'remark': 'Non-PD'},
 'N': {'pn': '35749091', 'remark': 'Non-PD'},
 'J': {'pn': '35930927', 'remark': 'PD'},
 'S': {'pn': '35930928', 'remark': 'PD'}}

In [5]:
def classify_barcode(barcode_information: str, remark_map: Dict[str, Dict[str, str]]) -> Dict[str, Optional[str]]:
    """barcode_information의 18번째 문자(1-indexed)를 key로 사용."""
    if barcode_information is None:
        return {"key18": None, "pn": None, "remark": None}
    s = str(barcode_information)
    key = s[17] if len(s) >= 18 else None  # 0-index 17 = 18th char
    if key is None:
        return {"key18": None, "pn": None, "remark": None}
    rec = remark_map.get(key)
    if not rec:
        return {"key18": key, "pn": None, "remark": None}
    return {"key18": key, "pn": rec.get("pn"), "remark": rec.get("remark")}

# 예시
example = "BA1WJ25273503681SJ8T-14F014-AE"
classify_barcode(example, remark_map)


{'key18': 'J', 'pn': '35930927', 'remark': 'PD'}

In [8]:
# =========================
# 4) station별 PASS / total / PASS% 산출
# =========================
# 대상:
# - 스키마: a1_fct_vision_testlog_txt_processing_history
# - 테이블: fct_vision_testlog_txt_processing_history
# 조건:
# - prod_day + shift_type에 해당하는 시간창만 스캔
# - station: 모두
# - goodorbad: 모두
# - result == 'PASS' 를 PASS로 집계

HIST_SCHEMA = "a1_fct_vision_testlog_txt_processing_history"
HIST_TABLE  = "fct_vision_testlog_txt_processing_history"

STATIONS = ["FCT1", "FCT2", "FCT3", "FCT4", "Vision1", "Vision2"]

def build_end_ts_expr(end_day_col: str = "end_day", end_time_col: str = "end_time") -> str:
    """end_day(YYYYMMDD) + end_time(HH:MI:SS[.fff]) => timestamp 로 합성.
    - end_time은 문자열로 캐스팅 후 소수점 이하 제거하여 'HH:MI:SS'로 정규화.
    """
    # split_part(end_time::text, '.', 1) => HH:MI:SS
    # end_day는 text로 캐스팅
    return (
        f"to_timestamp(({end_day_col})::text || ' ' || "
        f"split_part(({end_time_col})::text, '.', 1), 'YYYYMMDD HH24:MI:SS')"
    )

def query_station_passfail(engine, prod_day: str, shift_type: str) -> pd.DataFrame:
    start_dt, end_excl = shift_window_halfopen(prod_day, shift_type)
    end_ts_expr = build_end_ts_expr()

    sql = text(f'''
        WITH base AS (
            SELECT
                station::text AS station,
                result::text  AS result
            FROM {HIST_SCHEMA}.{HIST_TABLE}
            WHERE {end_ts_expr} >= :start_dt
              AND {end_ts_expr} <  :end_excl
        )
        SELECT
            station,
            COUNT(*) AS total_cnt,
            SUM(CASE WHEN result = 'PASS' THEN 1 ELSE 0 END) AS pass_cnt
        FROM base
        GROUP BY station
    ''')

    with engine.connect() as conn:
        df = pd.read_sql(sql, conn, params={"start_dt": start_dt, "end_excl": end_excl})

    # station 표준화
    df["station"] = df["station"].astype(str).str.strip()

    # 누락 station 보정
    full = pd.DataFrame({"station": STATIONS})
    df = full.merge(df, on="station", how="left").fillna({"total_cnt": 0, "pass_cnt": 0})
    df["total_cnt"] = df["total_cnt"].astype(int)
    df["pass_cnt"]  = df["pass_cnt"].astype(int)
    df["pass_pct"]  = df.apply(
        lambda r: (r["pass_cnt"] / r["total_cnt"] * 100.0) if r["total_cnt"] else 0.0,
        axis=1
    )
    return df

def format_cell(pass_cnt: int, total_cnt: int, pass_pct: float) -> str:
    return f"PASS : {pass_cnt:,}\n total : {total_cnt:,}\n PASS_pct:{pass_pct:0.2f}"

def build_report_row(prod_day: str, shift_type: str, station_df: pd.DataFrame) -> Dict[str, Any]:
    row = {"prod_day": prod_day, "shift_type": shift_type}
    mp = station_df.set_index("station")[["pass_cnt", "total_cnt", "pass_pct"]].to_dict("index")
    for st in STATIONS:
        rec = mp.get(st, {"pass_cnt": 0, "total_cnt": 0, "pass_pct": 0.0})
        row[st] = format_cell(int(rec["pass_cnt"]), int(rec["total_cnt"]), float(rec["pass_pct"]))
    return row

def run_report(engine, prod_days, shift_type: str) -> pd.DataFrame:
    rows = []
    for pd_day in prod_days:
        sdf = query_station_passfail(engine, pd_day, shift_type)
        rows.append(build_report_row(pd_day, shift_type, sdf))

    df = pd.DataFrame(rows, columns=["prod_day", "shift_type"] + STATIONS)

    # ✅ dataframe 생성 시각 timestamp (동일 시각을 모든 행에 기록)
    df["updated_at"] = pd.Timestamp.now()

    # ✅ updated_at을 제일 오른쪽으로 배치
    df = df[["prod_day", "shift_type"] + STATIONS + ["updated_at"]]
    return df

# =========================
# 실행 예시
# =========================
prod_days = ["20260127"]  # 여러 날짜도 가능: ["20260125","20260126","20260127"]
shift_type = "day"        # 'day' or 'night'

report_df = run_report(engine, prod_days, shift_type)
report_df

,prod_day,shift_type,FCT1,FCT2,FCT3,FCT4,Vision1,Vision2,updated_at
0,20260127,day,PASS : 940\n total : 945\n PASS_pct:99.47,"PASS : 1,011\n total : 1,015\n PASS_pct:99.61",PASS : 950\n total : 953\n PASS_pct:99.69,"PASS : 1,010\n total : 1,019\n PASS_pct:99.12","PASS : 1,936\n total : 1,945\n PASS_pct:99.54","PASS : 1,970\n total : 1,975\n PASS_pct:99.75",2026-01-29 17:15:05.908484


## Notes / 운영 팁

- **inclusive 시간 조건**(20:29:59 포함 등)을 정확히 맞추기 위해 SQL은 `>= start AND < end_exclusive` 형태로 처리했습니다.  
- `end_time`에 소수점(예: `12:01:08.64`)이 붙어도 `split_part(end_time::text, '.', 1)` 로 **HH:MI:SS** 로 정규화합니다.
- `remark_info` 테이블이 바뀌면 `load_remark_map()` 재실행으로 즉시 반영됩니다.
